# 06 · EDA step 5: pre-game predictors

**Question:** which pre-game features actually predict who wins, over which window, and
which of them are worth putting in the model?

**Why this matters for the model.** This is where the candidate list becomes a shortlist.
Steps 3 and 4 said which areas matter and which stats repeat. This step measures each
feature's predictive power on games it wasn't fit on, under rules fixed in advance, so the
shortlist reflects real signal and not a lucky split.

## Rules (fixed before running)

1. **Data:** 957 games (2025-26, D-I, no ties, both teams past the warm-up).
   `TimeSeriesSplit(5)` from step 1. **Folds 1–4 are for choosing; fold 5 is untouched**
   until the confirmation cell at the end.
2. **Score each feature alone:** a one-feature logistic regression fit on the training part
   of each fold (standardized inside the fold).
   - **gain** = baseline log loss − feature log loss on the fold's test games. The baseline
     always predicts the training home win rate.
   - Positive gain = the feature beat the baseline.
   - AUC is reported alongside.
3. **Keep a feature only if** its mean gain over folds 1–4 is above 0 **and** it beats the
   baseline in at least 3 of the 4 folds.
4. **Window choice** (tier 1 stats only):
   - Take the window with the best mean gain.
   - If the season-to-date window is within one standard error of the best, take the
     season instead, since it's simpler and steadier.
   - The standard error comes from the four paired fold differences.
5. **Overlap:** go down the kept features from strongest to weakest. Drop a feature whose
   correlation with one already kept is |r| ≥ 0.8, because it's mostly the same information.
6. **Confirm once on fold 5** (trained on everything before it). A shortlisted feature that
   doesn't beat the baseline there is **flagged**, not dropped. Step 7 treats flagged
   features as lower priority. Fold 5 includes the conference and NCAA tournaments.

Home ice, rest and back-to-backs are step 6. Here the baseline already includes the
average home edge.

In [ ]:
%run ./02_choose_k.ipynb

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

assert K == 5, "step 1 locked k = 5"
data = rows.reset_index(drop=True)
splits = list(TimeSeriesSplit(n_splits=K).split(data))
SELECT, CONFIRM = splits[:4], splits[4]
y = data["home_win"].to_numpy()
print(f"{len(data)} games, {len(FEATURES)} candidate features")
print(f"Choosing on folds 1-4 ({sum(len(t) for _, t in SELECT)} test games); "
      f"fold 5 ({len(CONFIRM[1])} games) held back")

## Score every feature on folds 1–4

In [ ]:
def score(feature, folds):
    out = []
    for train, test in folds:
        base = log_loss(y[test], np.full(len(test), y[train].mean()), labels=[0, 1])
        x = data[[feature]].to_numpy()
        m = make_pipeline(StandardScaler(), LogisticRegression()).fit(x[train], y[train])
        p = m.predict_proba(x[test])[:, 1]
        out.append({"gain": base - log_loss(y[test], p, labels=[0, 1]), "auc": roc_auc_score(y[test], p)})
    return pd.DataFrame(out)


per_fold = {f: score(f, SELECT) for f in FEATURES}


def parse(f):
    name = f.removeprefix("diff_")
    for w in WINDOWS:
        if name.endswith(f"_{w}"):
            return name[: -len(w) - 1], w
    return name, "season"  # sos, adj_goal_diff_pg, starter_save_pct, streak, form_delta


res = pd.DataFrame({
    f: {
        "stat": parse(f)[0], "window": parse(f)[1],
        "mean gain": s["gain"].mean(), "folds beating baseline": int((s["gain"] > 0).sum()),
        "worst fold": s["gain"].min(), "mean AUC": s["auc"].mean(),
    }
    for f, s in per_fold.items()
}).T
res["kept"] = (res["mean gain"].astype(float) > 0) & (res["folds beating baseline"] >= 3)
res = res.sort_values("mean gain", ascending=False)
print(f"Kept by rule 3: {int(res['kept'].sum())} of {len(res)}")
res.style.format({"mean gain": "{:+.4f}", "worst fold": "{:+.4f}", "mean AUC": "{:.3f}"})

## Does recent form beat the season?

Mean gain on folds 1–4 for every tier 1 stat by window. The row's best window is
**bold**.

In [ ]:
t1 = res[res["stat"].isin(TIER1)].pivot(index="stat", columns="window", values="mean gain").astype(float)
t1 = t1[["last3", "last5", "last10", "ewm", "season"]].loc[TIER1]
t1.style.format("{:+.4f}").highlight_max(axis=1, props="font-weight: bold")

In [ ]:
def paired_se(f1, f2):
    d = per_fold[f1]["gain"] - per_fold[f2]["gain"]
    return d.mean(), d.std(ddof=1) / np.sqrt(len(d))


choice = []
for stat in TIER1:
    best_w = t1.loc[stat].idxmax()
    best, season = f"diff_{stat}_{best_w}", f"diff_{stat}_season"
    diff, se = paired_se(best, season)
    pick = best_w if (best_w != "season" and diff > se) else "season"
    choice.append({"stat": stat, "best window": best_w, "best − season": diff, "SE": se,
                   "chosen": pick})
choice = pd.DataFrame(choice).set_index("stat")
choice.style.format({"best − season": "{:+.4f}", "SE": "{:.4f}"})

## Head-to-head questions from steps 3 and 4

Each pair uses the same folds, so the difference in mean gain is a fair comparison. The
**SE** is the standard error of that difference over the 4 folds.
- **Difference / SE above about 2:** a real edge.
- **Below 1:** a tie.

In [ ]:
def compare(a, b, label):
    d, se = paired_se(a, b)
    return {"question": label, "A": a.removeprefix("diff_"), "B": b.removeprefix("diff_"),
            "gain A": per_fold[a]["gain"].mean(), "gain B": per_fold[b]["gain"].mean(),
            "A − B": d, "SE": se, "diff / SE": d / se if se > 0 else np.nan}


h2h = pd.DataFrame([
    compare("diff_close_shot_share_season", "diff_shot_share_season", "Close-game vs raw shot share"),
    compare("diff_adj_goal_diff_pg", "diff_goal_diff_pg_season", "Schedule-adjusted vs raw goal diff"),
    compare("diff_goal_diff_pg_no_en_season", "diff_goal_diff_pg_season", "Goal diff without empty-netters vs with"),
    compare("diff_goal_diff_pg_season", "diff_win_pct_season", "Goal diff vs win %"),
    compare("diff_shot_share_season", "diff_goal_diff_pg_season", "Shot share vs goal diff"),
    compare("diff_st_index_season", "diff_pp_pct_season", "Special teams index vs PP%"),
    compare("diff_st_index_season", "diff_pk_pct_season", "Special teams index vs PK%"),
    compare("diff_starter_save_pct", "diff_save_pct_season", "Likely starter's save % vs team save %"),
    compare("diff_scored_first_pct_season", "diff_goal_diff_pg_season", "% scoring first vs goal diff"),
]).set_index("question")
h2h.style.format({"gain A": "{:+.4f}", "gain B": "{:+.4f}", "A − B": "{:+.4f}", "SE": "{:.4f}", "diff / SE": "{:+.1f}"})

## Shortlist

Rules 3–5 in order:
1. Only kept features.
2. Tier 1 stats at their chosen window.
3. Strongest first, skipping any feature with |r| ≥ 0.8 to one already on the list.

Correlations are measured on the games before fold 5.

In [ ]:
kept = res[res["kept"]].copy()
# rule 4: tier 1 stats only at their chosen window
is_t1 = kept["stat"].isin(TIER1)
kept = kept[~is_t1 | (kept["window"] == kept["stat"].map(choice["chosen"]))]

before_fold5 = data.iloc[: CONFIRM[1][0]]
corr = before_fold5[kept.index].corr()
shortlist, dropped = [], {}
for f in kept.sort_values("mean gain", ascending=False).index:
    clash = [g for g in shortlist if abs(corr.loc[f, g]) >= 0.8]
    if clash:
        dropped[f] = f"r = {corr.loc[f, clash[0]]:.2f} with {clash[0].removeprefix('diff_')}"
    else:
        shortlist.append(f)

print("Shortlist (strongest first):")
for f in shortlist:
    print(f"  {f.removeprefix('diff_'):<28} gain {res.loc[f, 'mean gain']:+.4f}, "
          f"beats baseline in {res.loc[f, 'folds beating baseline']}/4 folds")
print("\nKept but dropped for overlap:")
for f, why in dropped.items():
    print(f"  {f.removeprefix('diff_'):<28} {why}")

In [ ]:
BLUE, MUTED, GRID, INK = "#2a78d6", "#6b6a66", "#e1e0d9", "#1f1f1d"
show = res.loc[[*shortlist, *dropped]].sort_values("mean gain")
lo = [per_fold[f]["gain"].min() for f in show.index]
hi = [per_fold[f]["gain"].max() for f in show.index]
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(show) + 1.2))
yy = np.arange(len(show))
ax.hlines(yy, lo, hi, color=BLUE, lw=2, alpha=0.3)
on_list = [f in shortlist for f in show.index]
ax.plot(show["mean gain"].astype(float)[on_list], yy[on_list], "o", color=BLUE, ms=7, label="shortlist")
ax.plot(show["mean gain"].astype(float)[[not x for x in on_list]], yy[[not x for x in on_list]], "o",
        mfc="white", mec=BLUE, ms=7, label="dropped (overlap)")
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_yticks(yy, [f.removeprefix("diff_") for f in show.index])
ax.set_xlabel("log-loss gain over the baseline (dot = mean, line = range over folds 1-4)", color=INK)
ax.set_title("Kept features: how much each beats the baseline", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(axis="x", color=GRID, lw=0.6)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## Confirm once on fold 5

Train on every game before fold 5, test on fold 5 (Feb 21 – Apr 11, including the
conference and NCAA tournaments). This is the only time fold 5 is looked at. Nothing is
re-chosen after this.

In [ ]:
confirm = pd.DataFrame({
    f: score(f, [CONFIRM]).iloc[0] for f in shortlist
}).T
confirm["confirmed"] = confirm["gain"] > 0
confirm.index = [f.removeprefix("diff_") for f in confirm.index]
confirm.style.format({"gain": "{:+.4f}", "auc": "{:.3f}"})

## Implications for the model

**1. The season to date beats recent form, for every stat.**
- For all 7 tier 1 stats, the season window predicts best, followed by ewm, then last 10,
  last 5 and last 3, in that order every time.
- Recent form alone is noisier than the season figure and doesn't make up for it. Even
  "last 10" loses about a fifth of the signal.
- **For the model:** build on season-to-date values. Whether recent form adds anything *on
  top of* the season figure is a different question, which step 7 tests by adding ewm shot
  share to a season-based model. As a standalone input it's clearly worse.
- **For the dashboard:** the last-N view stays useful for scouting what a team is doing now,
  but for predicting the result, the season figure is the better guide.

**2. Shots predict best. Goal differential is close behind and says something extra.**
- Shot share alone cuts the baseline's log loss by about 10% (gain 0.070 on a baseline of
  about 0.69; AUC 0.71).
- Goal differential is close behind (0.057). The gap between them is within the noise
  (1.1 SE).
- The two overlap only partly, so both are on the shortlist. Step 7 checks how much they
  add together.

**3. Questions from earlier steps, answered:**

| Question | Answer | Choice |
|---|---|---|
| Close-game vs raw shot share | Tied (0.3 SE). Removing score effects didn't help prediction | Close-game is on the list by the rule; raw is interchangeable |
| Goal diff without empty-net goals | Tied (1.0 SE) | No-EN on the list by the rule; either works |
| Goal diff vs win % | **Goal diff better** (3.2 SE) | Win % dropped: it's a noisier copy |
| Schedule-adjusted goal diff | No better (−0.6 SE). Strength of schedule alone barely predicts | Raw goal diff; sos stays only as a weak candidate |
| Special teams index vs PP% / PK% | **Index better** than either (4.1 and 8.1 SE) | Index; PK% dropped as overlap, PP% kept as a weak extra |
| Likely starter's save % vs team save % | **Starter worse** (−3.1 SE) and fails the rule | Team save %. The starter guess or the 100-shot fallback adds noise |
| % scoring first vs goal diff | **Goal diff better** (−2.6 SE) | On the list (r < 0.8), but expected to add nothing in step 7 |

**4. Dropped by the rules:**
- Times shorthanded, PIM, the likely starter's save % and form vs. season. Each beat the
  baseline in only 1–2 of 4 folds.
- **Discipline doesn't predict at all**, as step 3 hinted.

**5. The shortlist for step 7** (strongest first):
- **Core:** close-game shot share, goal differential (no empty-net goals), shots for per
  game. Each beats the baseline in 4/4 folds, gain 0.048–0.071.
- **Secondary:** % scoring first, special teams index, streak, PP%. Gain 0.014–0.029.
- **Marginal:** save % (0.007), PP chances per game (0.003), sos (0.001). Their gains are
  tiny, close to noise.

**6. Fold 5 confirms the list.**
- 9 of 10 features beat the baseline on the held-back games.
- **PP chances per game is flagged:** it didn't confirm, and it was marginal anyway.
- Gains on fold 5 are smaller (0.044 for the best feature, against 0.071 on folds 1–4).
  Fold 5 is the conference and NCAA tournaments, with closer matchups and neutral sites, so
  expect lower accuracy at that time of year.

**7. What to expect from the final model.**
- The best single feature reaches AUC about 0.71 on folds 1–4 and 0.64 on the
  tournament-heavy fold 5.
- College hockey games are hard to call, and a combined model will add a little, not a lot.
- The realistic goal is honest, well-calibrated probabilities, not high accuracy.